# A2 · Confusion matrix, ROC curve and average precision from scratch

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Implement with numpy the metrics of lesson 4.3: confusion matrix, precision/recall/F1, ROC curve and AUC (in two ways), average precision and the
search for the threshold that maximizes F1, correctly handling **tied scores**. Compare everything with scikit-learn.

## Dataset

`scores` and `y_te`: the probabilities of a logistic on the test set of `ds.loans` (450 rows, about 19% positives), **rounded to 2 decimals** on purpose,
so that many examples share the same score.

## Educational purpose

The metrics are not formulas to memorize: they are counts on a moving threshold. Implementing them shows you that the AUC is the probability that a positive
beats a negative and, above all, that **ties** must be handled with care (it is the most common mistake when writing an ROC by hand).

## Task

1. `confusion_matrix_(y, pred)` → `(tn, fp, fn, tp)` and `metrics(y, pred)` → dictionary with `accuracy`, `precision`, `recall`, `f1` (0 if the denominator is 0).
2. `roc_curve_(y, s)` → `(fpr, tpr)`: sort the scores in decreasing order, accumulate true and false positives and **keep a single point for each distinct value** of the score (the point (0, 0) at the start). It must match `roc_curve(..., drop_intermediate=False)`.
3. Compute the AUC in two ways: `auc_trapezoids(fpr, tpr)` (area under the curve) and `auc_pairs(y, s)` (fraction of positive-negative pairs ordered correctly, with ties counting 1/2). They must match each other and `roc_auc_score`.
4. `average_precision(y, s)` = $\sum_k (R_k - R_{k-1}) P_k$ over the distinct values of the scores only (from high threshold to low); like `average_precision_score`.
5. `best_f1_threshold(y, s)` → `(threshold, f1)`: the threshold (among the distinct scores, with the rule `s >= threshold`) that maximizes F1.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
y_te = y_te.to_numpy()
scores = np.round(model.predict_proba(X_te)[:, 1], 2)
print(len(scores), "scores,", len(np.unique(scores)), "distinct values")

## Your work

In [ ]:
def confusion_matrix_(y, pred):
    ...


def metrics(y, pred):
    ...

In [ ]:
def roc_curve_(y, s):
    ...


def auc_trapezoids(fpr, tpr):
    ...


def auc_pairs(y, s):
    ...

In [ ]:
def average_precision(y, s):
    ...


def best_f1_threshold(y, s):
    ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`order = np.argsort(-s, kind="stable")`; `tps = np.cumsum(y[order])`; `fps = np.cumsum(1 - y[order])`; the indices of the **last** elements of each group of equal scores: `np.r_[np.where(np.diff(s[order]) != 0)[0], len(s) - 1]`. Pairs: `diff = pos[:, None] - neg[None, :]`.'):
    pred = (scores >= 0.3).astype(int)
    check(confusion_matrix_(y_te, pred) == tuple(confusion_matrix(y_te, pred).ravel()), "correct confusion matrix", "tn, fp, fn, tp: mind the order")
    m = metrics(y_te, pred)
    check(close([m["precision"], m["recall"], m["f1"]], [precision_score(y_te, pred), recall_score(y_te, pred), f1_score(y_te, pred)]), "correct precision, recall and F1", "precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = harmonic mean")
    check(m["accuracy"] == (pred == y_te).mean() and metrics(y_te, np.zeros_like(y_te))["precision"] == 0.0, "correct accuracy and no division by zero", "if you flag nothing the precision is 0 by convention")
    fpr, tpr = roc_curve_(y_te, scores)
    fpr_sk, tpr_sk, _ = roc_curve(y_te, scores, drop_intermediate=False)
    check(len(fpr) == len(np.unique(scores)) + 1, "one ROC point for every distinct value of the score (plus the origin)", "with ties you can't have a point for every example")
    check(close(fpr, fpr_sk) and close(tpr, tpr_sk), "the ROC curve matches scikit-learn's", "accumulate tp and fp only on the last indices of every group of equal scores")
    auc_sk = roc_auc_score(y_te, scores)
    check(close(auc_trapezoids(fpr, tpr), auc_sk) and close(auc_pairs(y_te, scores), auc_sk), "the AUC matches roc_auc_score in both ways", "pairs: (pos > neg) + 0.5 (pos == neg), divided by the number of pairs")
    check(close(average_precision(y_te, scores), average_precision_score(y_te, scores)), "correct average precision", "sum of (R_k - R_{k-1}) * P_k over the distinct values")
    s_best, f1_best = best_f1_threshold(y_te, scores)
    f1_brute_force = max(f1_score(y_te, (scores >= t).astype(int)) for t in np.unique(scores))
    check(close(f1_best, f1_brute_force) and close(f1_score(y_te, (scores >= s_best).astype(int)), f1_best), "the threshold found really maximizes F1", "try every distinct value of the score as a threshold")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_metrics_and_roc_curve_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.